In [1]:
import tvm
from tvm import relax
from tvm.relax.frontend.torch import from_exported_program

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import torchvision.models as models
from torchvision.io import read_image
from torch.export import export

import onnx
import os

# Model

In [2]:
class SoftmaxMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(128, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)

    def forward(self, x):
        x = self.fc1(x)
        x = F.softmax(x, dim=-1)

        x = self.fc2(x)
        x = F.softmax(x, dim=-1)

        x = self.fc3(x)
        x = F.softmax(x, dim=-1)

        return x
    
torch_model = SoftmaxMLP().eval()

IRModule

In [3]:
# Give an example argument to torch.export
example_args = (torch.randn(1, 128, dtype=torch.float32),)

# Convert the model to IRModule
with torch.no_grad():
    exported_program = export(torch_model, example_args)
    mod = from_exported_program(exported_program, keep_params_as_input=True)

mod, params = relax.frontend.detach_params(mod)
mod.show()

/usr/lib/python3/dist-packages/z3/z3core.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## Optimize model

In [4]:
TOTAL_TRIALS = 512
MAX_TRIALS_PER_TASK = 16 
target = tvm.target.Target({
    "kind": "llvm",
    "mtriple": "riscv64-linux-gnu",
    "mcpu": "generic-rv64",
    "mattr": ["+64bit", "+m", "+a", "+f", "+d", "+c", "+v"],
    "num-cores": 8,
})
work_dir = "tuning_logs"

mod = relax.get_pipeline(
    "static_shape_tuning",
    target=target,
    work_dir=work_dir,
    total_trials=TOTAL_TRIALS,
    max_trials_per_task=MAX_TRIALS_PER_TASK,
)(mod)

# Only show the main function
mod["main"].show()

2026-03-25 15:18:03 [INFO] Logging directory: tuning_logs/logs
2026-03-25 15:18:20 [INFO] LocalBuilder: max_workers = 8
2026-03-25 15:18:22 [INFO] LocalRunner: max_workers = 1


InternalError: Check failed: (it != type_key2index_.end()) is false: Cannot find type `tir.PrimFunc`

In [7]:
with target:
    mod = tvm.s_tir.transform.DefaultGPUSchedule()(mod)
ex = tvm.compile(mod, target=target)